# 03 — Tiling, parent grouping, and the fold protocol

**What this notebook does.** Turns 1,498 image/boundary pairs into a tile
index, groups every tile under the micrograph it was cut from, and writes the
fold manifests the training step will read. It cuts 256 px tiles at stride 128
(50% overlap), reflection-padding only the right and bottom edges where the
last tile overruns; nothing is ever resized. It then builds the split
protocol: Steel2 held out as a test-only domain-shift fold, Steel1 split by
parent, and leave-one-dataset-out across MetalDam, uhcs1 and uhcs2.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted — `00_bootstrap.ipynb` passes
- `reports/audit.json` — where each dataset lives, from `01_audit.ipynb`
- `reports/gt_extraction.json` and the boundary PNGs under
  `PERSISTENT_DIR/gt_boundaries/` — from `02_boundary_gt.ipynb`

**What it produces.** `reports/manifests/*.csv` (one per fold plus the test
manifest), `configs/fold_stats.yaml` (sampling weights and `pos_weight` for
the training sampler), `reports/parents.md` (the grouping, checkable by eye)
and `reports/tiling.{md,json}`. No tile images are written: a tile is a row in
a manifest and the loader crops it on the fly, so the pixels exist once.

**Expected runtime on a free T4.** 2–4 minutes. Every boundary PNG is opened
once to measure per-tile boundary fraction; no GPU is used.

## Cell 1 — the standard bootstrap block

Identical in every notebook. Reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py` through the GitHub API, then hands over
to `bootstrap()`, which clones the repo, installs what is missing, mounts
Drive on Colab, verifies `DATA_ROOT` and returns `PATHS`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Why tiles, and why parents

**Why tiling replaces resizing.** A micrograph is a physical measurement: each
pixel covers a known number of microns, and the width of a grain boundary in
pixels is a real quantity the network has to learn. Resizing an image to a
fixed input size silently rescales that quantity — a 2 px boundary becomes
1.4 px in one dataset and 3 px in another, and the network is asked to call
both the same thing. Worse, the five datasets here differ in size (1280×895
down to 256×256), so a single resize target would stretch some and squash
others by different factors. Cutting fixed 256 px tiles keeps every pixel at
its native scale.

The last tile of each row and column is **clamped to the image edge** rather
than allowed to overrun: for a 645 px axis the offsets are 0, 128, 256, 389,
and the final tile ends exactly at 645. The alternative — letting it start at
512 and reflection-padding the missing 123 px — would have made 48% of that
tile mirrored texture, microstructure that exists in no micrograph, with an
artificial mirror-symmetric boundary straight down the seam. The network would
have learned that seam as a feature. Clamping costs only an uneven overlap at
one position, which is harmless, and fabricates nothing. Padding survives for
exactly one case, an image smaller than the patch in an axis; no dataset here
has one, and the report counts every such pixel so it cannot slip in unnoticed.

Patch size is 256 and not 512 for a measured reason: Steel1 and Steel2 ship
pre-tiled at exactly 256×256. A 512 patch could not be cut from either without
padding half of every tile with invented pixels — 1,411 of the 1,498 images.

**Why parent grouping is mandatory.** Steel1's 907 tiles come from **19**
source micrographs, 48 tiles each; Steel2's 504 come from a handful of
originals. Tiles from one micrograph are near-duplicates: the same specimen,
the same etch, the same illumination, and at stride 128 each tile shares half
its pixels with its neighbour.

Put one such tile in train and its neighbour in val, and the validation score
stops measuring generalisation. The model can score well by memorising the
specimen it was trained on — the val tile is 128 px away from a tile it has
already seen, sharing literally half its pixels. The number it produces would
look excellent and mean nothing, and the failure is invisible: no error is
raised, the curve just looks better than the model is.

So every tile carries a `parent_id` — the source micrograph, recovered by
stripping the tile index from the filename — and **splits are made by parent,
never by tile**. The checks cell asserts that no parent appears on both sides
of any fold, because this is the kind of mistake that is silent until the
model reaches real data.

## Build the tile index and the parent map

`src/tiling.py` does the work: it reads `reports/gt_extraction.json` for the
boundary maps (and for the exact crops step 2 applied to the two reconciled
MetalDam pairs — those crops are *reused*, never re-derived, so a pair cannot
end up misaligned by a row), plans the tile grid per image, measures each
tile's boundary fraction from the boundary PNG, and drops tiles below
`min_boundary_frac`.

The printout below is the structure you should be able to recognise: Steel1 at
19 parents × 48 tiles, Steel2 at one tile per image, and the three SEM sets
with one parent per image and many tiles each. If Steel1 does not say 19, the
exclusion list or the parent rule is wrong and the checks cell will fail.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src import boundary_gt, tiling

settings = tiling.load_config()
reports_dir = Path(PATHS["reports_dir"])
gt_root = Path(PATHS["gt_boundaries_root"])
manifest_dir = reports_dir / settings["manifest_subdir"]

audit = boundary_gt.load_audit(reports_dir)
extraction = tiling.load_extraction(reports_dir)

print(f"patch {settings['patch_size']} px, stride {settings['stride']} px, "
      f"min boundary fraction {settings['min_boundary_frac']}")
print(f"boundary maps: {gt_root}\n")


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False, unit="img")


index = tiling.build_index(extraction, audit, gt_root, settings, progress=progress)
tiles = tiling.all_tiles(index)
pmap = tiling.parent_map(tiles)

rows = []
for name, d in index["datasets"].items():
    rows.append({
        "dataset": name,
        "parents": d["n_parents"],
        "images": d["n_images"],
        "tiles": d["n_tiles"],
        "fabricated px": d["padded_pixels"],
        "dropped (low boundary)": d["n_dropped"],
        "excluded": d["n_excluded"],
        "images with no mask": d["n_orphans"],
        "no boundary map": d["n_skipped"],
    })
display(pd.DataFrame(rows).set_index("dataset"))

for name, parents in pmap.items():
    counts = [p["tiles"] for p in parents.values()]
    print(f"\n{name}: {len(parents)} parents, tiles per parent "
          f"min/median/max = {min(counts)}/{int(np.median(counts))}/{max(counts)}")
    if len(parents) <= 25:
        for parent, p in parents.items():
            print(f"    {parent:<45} {p['n_images']:>3} image(s)  {p['tiles']:>4} tiles")
    else:
        print(f"    (one parent per source image; not listed)")

fabricated = sum(d["padded_pixels"] for d in index["datasets"].values())
print(f"\nfabricated (padded) pixels across every dataset: {fabricated}")
for name, d in index["datasets"].items():
    for x in d["padded_images"]:
        print(f"!! {name}/{x['image']} is {x['size'][0]}x{x['size'][1]}, smaller "
              f"than the {settings['patch_size']} px patch: {x['padded_pixels']} "
              f"pixels padded ({x['why']})")

for d in index["datasets"].values():
    for loud in d["single_tile_drops"]:
        print(f"\n!! {loud['dataset']}/{loud['image']} lost its ONLY tile "
              f"(boundary fraction {loud['boundary_fraction']:.5f}) — "
              f"the image is absent from every split")
    if d["listed_but_absent"]:
        print(f"\n!! {d['dataset']}: tiling.exclusions names files that do not "
              f"exist: {d['listed_but_absent']}")

## Build the folds

The protocol, and why each part of it is there:

- **Steel2 is test-only.** It is the one optical/colour dataset among four
  grayscale SEM sets, and its masks are auto-thresholded rather than
  hand-curated — its anti-aliased edge ramp is the giveaway. Training on it
  would teach threshold artifacts as structure; held out, it measures whether
  the model survives a domain shift, which is exactly what a prior generator
  for Phase 1 must do.
- **Steel1 is split by parent**, 15 of its 19 parents to train and 4 to val,
  chosen once with a fixed seed. Because Steel1 is 91% of the tile count, it
  cannot simply be held out or included wholesale: it appears on both sides of
  every fold, but never with the same specimen on both.
- **MetalDam, uhcs1 and uhcs2 rotate** as the held-out validation dataset, one
  per fold — leave-one-*dataset*-out, the only split that measures transfer to
  an unseen imaging setup.

Each fold is therefore `val = {held-out dataset} + {Steel1 val parents}` and
`train = {the other two datasets} + {Steel1 train parents}`. `dev` is an alias
for whichever fold has the smallest validation set, for fast iteration — the
same protocol, less waiting.

In [ ]:
folds = tiling.build_folds(index, settings)
stats = tiling.fold_statistics(folds, settings)

manifests = tiling.write_manifests(folds, manifest_dir)
parents_path = tiling.write_parents_md(index, reports_dir)
md_path, json_path = tiling.write_tiling_report(
    index, folds, stats, manifests, reports_dir)
stats_path = tiling.write_fold_stats(stats, folds, settings)

print("Steel1 parent split (fixed, seed "
      f"{settings['seed']}):")
for side, names in folds["parent_splits"].get("Steel1", {}).items():
    print(f"  {side:<5} {len(names):>2} parents: {names}")

for path in (parents_path, md_path, json_path, stats_path):
    print(f"wrote {path}")
for name, path in manifests.items():
    print(f"wrote {path}")

## What each fold actually contains

One row per fold: how many tiles on each side, which datasets they come from,
how many independent parents that really is, the mean boundary fraction, and
`pos_weight` — the ratio of background to boundary pixels in the training
split, which is what `BCEWithLogitsLoss` needs to stop the network from
predicting "no boundary" everywhere.

Read the parent counts, not the tile counts. A fold with 1,500 training tiles
from 30 parents has 30 independent scenes; that is the number that governs
whether the model generalises.

In [ ]:
rows = []
for name, f in stats.items():
    if name == "test":
        continue
    rows.append({
        "fold": name + (f" (= {f['alias_of']})" if f.get("alias_of") else ""),
        "held out": f["held_out"],
        "train tiles": f["n_train_tiles"],
        "val tiles": f["n_val_tiles"],
        "train parents": f["n_train_parents"],
        "val parents": f["n_val_parents"],
        "train datasets": ", ".join(f["train_datasets"]),
        "val datasets": ", ".join(f["val_datasets"]),
        "train frac (mean)": f["train_boundary_fraction"]["mean"],
        "val frac (mean)": f["val_boundary_fraction"]["mean"],
        "pos_weight": f["pos_weight"],
    })
display(pd.DataFrame(rows).set_index("fold"))

t = stats["test"]
print(f"test manifest: {', '.join(t['datasets'])} — {t['n_tiles']} tiles from "
      f"{t['n_parents']} parents, mean boundary fraction "
      f"{t['boundary_fraction']['mean']}")
print("Steel2 appears in no train or val split of any fold.")

## The rebalancing, drawn

Raw tile counts do not measure independent information. Steel1 contributes
~907 tiles from 19 micrographs; uhcs2 contributes ~345 from 23. Sampled by
raw count, an epoch would be dominated by one specimen set and the loss would
follow it.

`weight_mode: parents` gives each dataset a share of the epoch equal to its
share of the independent *scenes*, and the per-tile weight that achieves it is
written to `configs/fold_stats.yaml` for a `WeightedRandomSampler`. The bars
below are that correction: raw tiles on the left of each pair, effective tiles
per epoch on the right. The epoch keeps its size; only the mix changes.

In [ ]:
import matplotlib.pyplot as plt

fold_names = [n for n in stats if n != "test"]
fig, axes = plt.subplots(1, len(fold_names), figsize=(5.2 * len(fold_names), 4.4),
                         squeeze=False)
for ax, name in zip(axes[0], fold_names):
    sm = stats[name]["sampling"]
    datasets = list(sm["raw_counts"])
    raw = [sm["raw_counts"][d] for d in datasets]
    eff = [sm["effective_counts"][d] for d in datasets]
    pos = np.arange(len(datasets))
    ax.bar(pos - 0.2, raw, 0.4, label="raw tiles", color="#7f9fc4")
    ax.bar(pos + 0.2, eff, 0.4, label="effective / epoch", color="#c47f7f")
    for i, d in enumerate(datasets):
        ax.text(i, max(raw[i], eff[i]), f"\n{sm['n_parents'][d]} parents",
                ha="center", va="bottom", fontsize=7)
    ax.set_xticks(pos)
    ax.set_xticklabels(datasets, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{name} train split\nweight_mode = {sm['mode']}", fontsize=10)
    ax.set_ylabel("tiles")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Where the boundary-fraction threshold falls

Every tile below `min_boundary_frac` is dropped: a tile that is 99.9%
background teaches the network only to say "background", and there are enough
of them in Steel1's flatter tiles to bias the loss.

The histograms below include the dropped tiles, so the threshold line shows
what was actually cut rather than what survived. A dataset with a large mass
to the left of the line is one where tiling is producing many empty crops —
worth knowing before training, not after.

In [ ]:
thresh = float(settings["min_boundary_frac"])
names = list(index["datasets"])
fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 3.8),
                         squeeze=False)
for ax, name in zip(axes[0], names):
    d = index["datasets"][name]
    kept = [t["boundary_fraction"] for t in d["tiles"]]
    dropped = [x["boundary_fraction"] for x in d["dropped"]]
    if not kept and not dropped:
        ax.set_title(f"{name}: no tiles")
        continue
    ax.hist([kept, dropped], bins=40, stacked=True,
            color=["#5b8c5a", "#c0703f"], label=["kept", "dropped"])
    ax.set_yscale("log")
    ax.axvline(thresh, color="k", linestyle="--", linewidth=1)
    ax.text(thresh, ax.get_ylim()[1], f" min {thresh}", fontsize=7, va="top")
    ax.set_title(f"{name}\n{len(kept)} kept, {len(dropped)} dropped", fontsize=10)
    ax.set_xlabel("tile boundary fraction")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## The tile grid, drawn over real images

Coordinates in a manifest are easy to get subtly wrong, so this cell draws the
grid it actually produced over one MetalDam image and one uhcs1 image.

What to check: tiles step by half a tile, and the **last** column and row are
clamped flush to the right and bottom edges — drawn in orange, with their
offset from the regular stride printed in the title. That uneven overlap is
the whole cost of not fabricating pixels: for uhcs1 the last column starts at
389 rather than 384, overlapping its neighbour by 5 px more than the others.

The grid must cover every pixel of the image with nothing hanging over the
edge. `fabricated px` in the title is the count of padded pixels for that
image, and it should read 0 for every dataset in this collection.

In [ ]:
from src import audit as audit_mod

SHOW = ["MetalDam", "uhcs1"]

fig, axes = plt.subplots(1, len(SHOW), figsize=(8.5 * len(SHOW), 7), squeeze=False)
for ax, name in zip(axes[0], SHOW):
    d = index["datasets"].get(name)
    if not d or not d["tiles"]:
        ax.set_title(f"{name}: no tiles")
        continue
    first = d["tiles"][0]
    image = audit_mod.read_array(Path(first["image_path"]))
    if first["crop_width"] != image.shape[1] or first["crop_height"] != image.shape[0]:
        image = image[first["crop_top"]:first["crop_top"] + first["crop_height"],
                      first["crop_left"]:first["crop_left"] + first["crop_width"]]
    plan = tiling.plan_tiles(image.shape[1], image.shape[0], settings)
    patch = plan["patch"]
    print(f"{name}: xs={plan['xs']}  ys={plan['ys']}  "
          f"covers {plan['xs'][-1] + patch}x{plan['ys'][-1] + patch} of "
          f"{image.shape[1]}x{image.shape[0]}  fabricated px "
          f"{plan['padded_pixels']}")

    stride = plan["stride"]
    clamped_x = plan["xs"][-1] != (len(plan["xs"]) - 1) * stride
    clamped_y = plan["ys"][-1] != (len(plan["ys"]) - 1) * stride

    ax.imshow(image, cmap="gray" if image.ndim == 2 else None,
              interpolation="nearest")
    for j, y in enumerate(plan["ys"]):
        for k, x in enumerate(plan["xs"]):
            edge_tile = (clamped_x and k == len(plan["xs"]) - 1) or \
                        (clamped_y and j == len(plan["ys"]) - 1)
            ax.add_patch(plt.Rectangle(
                (x, y), patch, patch, fill=False,
                edgecolor="#ff8c00" if edge_tile else "#ffd400",
                linewidth=1.6 if edge_tile else 0.8, alpha=0.95))

    shift_x = plan["xs"][-1] - (len(plan["xs"]) - 1) * stride
    shift_y = plan["ys"][-1] - (len(plan["ys"]) - 1) * stride
    ax.set_xlim(-8, image.shape[1] + 8)
    ax.set_ylim(image.shape[0] + 8, -8)
    ax.set_title(
        f"{name} · {first['source_image']} · {image.shape[1]}x{image.shape[0]}\n"
        f"{len(plan['xs'])}x{len(plan['ys'])} = {plan['n_tiles']} tiles of "
        f"{patch} px, stride {stride}\n"
        f"last column clamped to x={plan['xs'][-1]} ({shift_x:+d} px), "
        f"last row clamped to y={plan['ys'][-1]} ({shift_y:+d} px), "
        f"fabricated px {plan['padded_pixels']}",
        fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Checks

Nothing here runs locally, so this cell is where the step is declared correct.
It re-reads the manifest CSVs **from disk** rather than trusting the in-memory
objects, because the CSVs are what the training step will consume.

The leakage checks are the important ones. A parent on both sides of a fold,
or a Steel2 tile in a train split, produces no error at training time — it
produces a validation number that looks good and is worthless. These are
exactly the failures that must be caught here, mechanically, and not by
reading a curve later.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


loaded = {name: pd.read_csv(path) for name, path in manifests.items()}
patch = int(settings["patch_size"])
lo_pw, hi_pw = tiling.SANE_POS_WEIGHT_BAND
excluded_images = set((settings["exclusions"] or {}).get("Steel1", []))
orphan_parent = "im_val_cut10-400-01-V-9mm-4000x13"

check("every manifest written and non-empty",
      all(not df.empty for df in loaded.values()),
      ", ".join(f"{n}={len(df)}" for n, df in loaded.items()))

fabricated = {name: d["padded_pixels"] for name, d in index["datasets"].items()}
check("no fabricated pixels: every tile is real image data",
      not any(fabricated.values()),
      ", ".join(f"{k}={v}" for k, v in fabricated.items()))

for name, df in loaded.items():
    covered = ((df["pad_right"] == 0) & (df["pad_bottom"] == 0)).all()
    check(f"{name}: manifest records no padding", bool(covered),
          "all rows pad 0,0" if covered else "padding present")

steel1 = index["datasets"].get("Steel1")
check("Steel1 resolves to exactly 19 parents",
      steel1 is not None and steel1["n_parents"] == 19,
      f"{steel1['n_parents'] if steel1 else 'missing'} parents")

orphans_present = {
    name: sorted((set(df["source_image"]) & excluded_images)
                 | set(df.loc[df["parent_id"] == orphan_parent, "source_image"]))
    for name, df in loaded.items()
}
check("the 48 orphan Steel1 tiles are in no manifest",
      all(not v for v in orphans_present.values()),
      f"{len(excluded_images)} excluded images, none present"
      if all(not v for v in orphans_present.values())
      else f"found {orphans_present}")

for name, df in loaded.items():
    if name == "test":
        continue
    train_p = set(map(tuple, df.loc[df["split"] == "train",
                                    ["dataset", "parent_id"]].values))
    val_p = set(map(tuple, df.loc[df["split"] == "val",
                                  ["dataset", "parent_id"]].values))
    both = train_p & val_p
    check(f"{name}: no parent in both train and val", not both,
          f"{len(train_p)} train / {len(val_p)} val parents"
          if not both else f"LEAKED: {sorted(both)[:5]}")

for name, df in loaded.items():
    if name == "test":
        continue
    check(f"{name}: Steel2 absent from train and val",
          "Steel2" not in set(df["dataset"]),
          f"datasets = {sorted(set(df['dataset']))}")
check("Steel2 is present in the test manifest",
      set(loaded["test"]["dataset"]) == {"Steel2"},
      f"{len(loaded['test'])} tiles, datasets {sorted(set(loaded['test']['dataset']))}")

pretiled = [d for d in (settings["assert_single_tile"] or [])]
for name, df in loaded.items():
    sub = df[df["dataset"].isin(pretiled)]
    if sub.empty:
        continue
    ok = bool((sub["crop_width"] == patch).all() and (sub["crop_height"] == patch).all()
              and (sub["pad_right"] == 0).all() and (sub["pad_bottom"] == 0).all()
              and (sub["x"] == 0).all() and (sub["y"] == 0).all())
    per_image = sub.groupby("source_image").size()
    check(f"{name}: {'/'.join(pretiled)} tiles are {patch}x{patch}, one per image, no padding",
          ok and int(per_image.max()) == 1,
          f"{len(sub)} tiles, max {int(per_image.max())} per image")

for name, df in loaded.items():
    within = ((df["x"] >= 0) & (df["y"] >= 0)
              & (df["x"] + patch <= df["crop_width"] + df["pad_right"])
              & (df["y"] + patch <= df["crop_height"] + df["pad_bottom"]))
    check(f"{name}: every tile lies inside its image plus padding",
          bool(within.all()),
          f"{len(df)} rows" if within.all() else f"{int((~within).sum())} out of bounds")

for name, f in stats.items():
    if name == "test":
        continue
    pw = f["pos_weight"]
    check(f"{name}: pos_weight finite and in {lo_pw}-{hi_pw}",
          pw is not None and np.isfinite(pw) and lo_pw <= pw <= hi_pw,
          f"pos_weight = {pw}")

check("fold_stats.yaml written", stats_path.is_file(), str(stats_path.name))
check("parents.md written", parents_path.is_file(), str(parents_path.name))

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Push the manifests, the fold statistics and the reports

The manifests *are* the dataset definition for step 4: which tile, from which
parent, on which side of which fold. They only become that once they are in
the repo, so this cell stages `reports/` (manifests, `parents.md`,
`tiling.md`, `tiling.json`) and `configs/` (`fold_stats.yaml`) and pushes.
Nothing under `data/` or `PERSISTENT_DIR` is touched.

In [ ]:
import subprocess

import yaml

from scripts.push_results import push_results

push_results(
    "step 3: tiling, parent grouping, hybrid fold protocol",
    paths=PATHS,
    expect=[md_path, json_path, stats_path, parents_path] + list(manifests.values()),
)

# The repo copy is what step 4 will read, so verify it matches THIS run rather
# than trusting that the push happened. A committed file that disagrees with
# the numbers just computed means the working tree was reset after the run (a
# re-run of the bootstrap cell does exactly that) and the manifests in the repo
# are stale -- which would train the next step on the wrong splits, silently.
repo_root = Path(PATHS["repo_root"])
committed = subprocess.run(
    ["git", "show", "HEAD:configs/fold_stats.yaml"],
    cwd=repo_root, capture_output=True, text=True)
if committed.returncode != 0:
    raise AssertionError("configs/fold_stats.yaml is not committed at HEAD; the "
                         "manifests never reached the repo.")

doc = yaml.safe_load(committed.stdout)
mismatch = {
    name: (f["pos_weight"], doc["folds"].get(name, {}).get("pos_weight"))
    for name, f in stats.items()
    if name != "test" and doc["folds"].get(name, {}).get("pos_weight") != f["pos_weight"]
}
n_tiles_now = sum(d["n_tiles"] for d in index["datasets"].values())
dev_committed = doc["folds"].get("dev", {})
n_tiles_committed = (dev_committed.get("n_train_tiles", 0)
                     + dev_committed.get("n_val_tiles", 0))

print(f"this run: {n_tiles_now} tiles indexed "
      f"({stats['dev']['n_train_tiles'] + stats['dev']['n_val_tiles']} in the dev fold)")
print(f"committed dev fold: {n_tiles_committed} tiles")
print(f"committed fold_stats.yaml: {doc['generated_utc']}, weight_mode "
      f"{doc['weight_mode']}, patch {doc['patch_size']}")
if mismatch:
    raise AssertionError(
        "the committed fold_stats.yaml does not match this run "
        f"(fold: this run vs committed) {mismatch}. The repo holds STALE "
        "manifests; re-run this notebook's cells from the tile index onward "
        "and push again before starting step 4."
    )
print("PASS  the committed manifests and fold statistics match this run")